# Augment and Dedupe Receipts (one-time migration)

Backfills missing `event_ts` / `event_date` columns on legacy Silver receipt
headers, lines, payments, and online-order headers. It also enforces the
business keys for receipt headers, receipt lines, and payments.

## Safety contract
- Exact duplicate headers are collapsed by `receipt_id_ext`; conflicting
  headers fail closed instead of choosing an arbitrary transaction.
- Exact duplicate lines and payments are collapsed by
  (`receipt_id_ext`, `line_num`) and `transaction_id`, respectively.
- Every candidate is materialized and validated in a run-scoped staging
  schema before promotion. Existing targets are backed up and restored if
  promotion or post-promotion validation fails.

The legacy receipt identifier format encodes the event time:
```
  RCP YYYYMMDDHHMM <store_id padded> <sequence>
  e.g. RCP 202412011318 01 25587
```
The timestamp is parsed from positions 4-15 and propagated from each validated
header to its lines and payments.

## Streaming guard
The repair replaces Silver receipt tables with `CREATE OR REPLACE`. Rows that
notebook 03 merges while it runs would be lost, so the notebook refuses to run
once streaming has initialized (`ag._watermarks` exists), exactly like the
historical rebuild guard (`writer.assert_historical_replacement_safe`). It
also holds the `silver` writer lock (`ag._job_locks`) while it runs.

## Usage
Run **once** in Fabric against the `retail_lakehouse` Lakehouse, before
streaming starts, then refresh the `retail_model` semantic model. The repair
is idempotent.

In [ ]:
import os
import uuid

from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

In [ ]:
LAKEHOUSE_NAME = os.environ.get("LAKEHOUSE_NAME", "retail_lakehouse")
SILVER_DB = os.environ.get("SILVER_DB", "ag")
SILVER_STAGE_DB = os.environ.get("SILVER_STAGE_DB", f"{SILVER_DB}_stage")
RUN_TOKEN = uuid.uuid4().hex

RECEIPTS_TABLE = f"{LAKEHOUSE_NAME}.{SILVER_DB}.fact_receipts"
LINES_TABLE = f"{LAKEHOUSE_NAME}.{SILVER_DB}.fact_receipt_lines"
PAYMENTS_TABLE = f"{LAKEHOUSE_NAME}.{SILVER_DB}.fact_payments"
ONLINE_ORDERS_TABLE = (
    f"{LAKEHOUSE_NAME}.{SILVER_DB}.fact_online_order_headers"
)

WATERMARK_TABLE = f"{LAKEHOUSE_NAME}.{SILVER_DB}._watermarks"
LOCK_WAIT_SECONDS = int(os.environ.get("LOCK_WAIT_SECONDS", "600"))


def assert_streaming_not_initialized():
    """Same guard as writer.assert_historical_replacement_safe."""
    if spark.catalog.tableExists(WATERMARK_TABLE):
        raise RuntimeError(
            f"Receipt replacement is blocked because {WATERMARK_TABLE} exists: "
            "notebook 03 merges live rows into these tables. Stop streaming and use "
            "the documented coordinated Silver/checkpoint reset first; never delete "
            "checkpoints alone."
        )


assert_streaming_not_initialized()
spark.sql(
    f"CREATE DATABASE IF NOT EXISTS {LAKEHOUSE_NAME}.{SILVER_STAGE_DB}"
)
print(
    f"Targeting {LAKEHOUSE_NAME}.{SILVER_DB}; "
    f"staging in {SILVER_STAGE_DB} for run {RUN_TOKEN}"
)

In [ ]:
# SHARED SOURCE (generated - do not edit): retail_setup/generation/job_locks.py
# Edit utility/src/retail_setup/generation/job_locks.py, then run utility/scripts/build_notebooks.py.

"""Lease locks that keep Lakehouse writers from overlapping.

``streaming-data-load`` (03, 04), ``daily-maintenance`` (05 and the ML
refresh), ``ml-required`` (setup-11 and the ML producers) and notebook 90 run
on independent schedules and write the same Delta tables. Fabric has no
cross-pipeline mutex, so each writer claims a lease row in
``<silver>._job_locks`` before it writes:

- ``silver``: notebook 03 and notebook 90 (all Silver tables and ``_watermarks``);
- ``gold``: notebook 04 and setup-11 (the ten Gold aggregates, ``setup_run_log``);
- ``ml:<table>``: the producer of that ML output table.

Notebook 05 claims the same lock for each table it maintains, without
waiting, and skips the table when another job holds it.

A claim is a Delta ``MERGE`` that succeeds only when the row is missing,
expired, or already ours; concurrent claims conflict in Delta and one of them
loses. The holder renews its lease from a background heartbeat, so a crashed
notebook releases its locks once the lease expires (``JOB_LOCK_TTL_SECONDS``).
All times come from Spark's ``current_timestamp()``, never the driver clock.

Keep this module free of package imports: Lakehouse notebooks run it as a
cell (``scripts/build_notebooks.py`` copies it in).
"""

import threading
import time as _time
import uuid
from collections.abc import Callable, Iterable
from datetime import datetime, timedelta, timezone

JOB_LOCK_TABLE = "_job_locks"
JOB_LOCK_TTL_SECONDS = 900
JOB_LOCK_HEARTBEAT_SECONDS = 120
JOB_LOCK_POLL_SECONDS = 30
SILVER_LOCK = "silver"
GOLD_LOCK = "gold"


def ml_output_lock(table: str) -> str:
    """The lock that guards one ML output table."""
    return f"ml:{table}"


def is_concurrent_write_error(exc: BaseException) -> bool:
    """True for Delta optimistic-concurrency conflicts (another writer won)."""
    text = f"{type(exc).__name__}: {exc}"
    return "Concurrent" in text or "DELTA_CONCURRENT" in text


class DeltaJobLockStore:
    """Lease rows in a Delta table: (lock_name, owner, acquired_at, expires_at)."""

    def __init__(self, spark, table_name: str, ttl_seconds: int = JOB_LOCK_TTL_SECONDS):
        self.spark = spark
        self.table_name = table_name
        self.ttl_seconds = int(ttl_seconds)
        self.spark.sql(
            f"CREATE TABLE IF NOT EXISTS {table_name} "
            "(lock_name STRING, owner STRING, acquired_at TIMESTAMP, expires_at TIMESTAMP) "
            "USING DELTA"
        )

    @staticmethod
    def _quote(value: str) -> str:
        return "'" + value.replace("\\", "\\\\").replace("'", "\\'") + "'"

    def holder(self, name: str) -> str | None:
        """The owner of a live lease on ``name``, or None."""
        rows = self.spark.sql(
            f"SELECT owner FROM {self.table_name} "
            f"WHERE lock_name = {self._quote(name)} AND expires_at > current_timestamp()"
        ).collect()
        return rows[0]["owner"] if rows else None

    def try_claim(self, name: str, owner: str) -> bool:
        try:
            self.spark.sql(
                f"""
                MERGE INTO {self.table_name} AS t
                USING (
                    SELECT {self._quote(name)} AS lock_name, {self._quote(owner)} AS owner,
                           current_timestamp() AS acquired_at,
                           current_timestamp() + INTERVAL {self.ttl_seconds} SECONDS AS expires_at
                ) AS s
                ON t.lock_name = s.lock_name
                WHEN MATCHED AND (t.expires_at <= s.acquired_at OR t.owner = s.owner)
                    THEN UPDATE SET t.owner = s.owner, t.acquired_at = s.acquired_at,
                                    t.expires_at = s.expires_at
                WHEN NOT MATCHED THEN INSERT (lock_name, owner, acquired_at, expires_at)
                    VALUES (s.lock_name, s.owner, s.acquired_at, s.expires_at)
                """
            )
        except Exception as exc:
            if is_concurrent_write_error(exc):
                return False
            raise
        return self.holder(name) == owner

    def renew(self, name: str, owner: str) -> bool:
        try:
            self.spark.sql(
                f"UPDATE {self.table_name} "
                f"SET expires_at = current_timestamp() + INTERVAL {self.ttl_seconds} SECONDS "
                f"WHERE lock_name = {self._quote(name)} AND owner = {self._quote(owner)}"
            )
        except Exception as exc:
            if not is_concurrent_write_error(exc):
                raise
        return self.holder(name) == owner

    def release(self, name: str, owner: str) -> None:
        for _attempt in range(5):
            try:
                self.spark.sql(
                    f"DELETE FROM {self.table_name} "
                    f"WHERE lock_name = {self._quote(name)} AND owner = {self._quote(owner)}"
                )
                return
            except Exception as exc:
                if not is_concurrent_write_error(exc):
                    raise
                _time.sleep(1)


class MemoryJobLockStore:
    """The same lease semantics in memory, with an injectable clock (tests)."""

    def __init__(
        self,
        ttl_seconds: int = JOB_LOCK_TTL_SECONDS,
        clock: Callable[[], datetime] | None = None,
    ):
        self.ttl = timedelta(seconds=ttl_seconds)
        self.clock = clock or (lambda: datetime.now(timezone.utc).replace(tzinfo=None))
        self.rows: dict[str, tuple[str, datetime]] = {}

    def holder(self, name: str) -> str | None:
        row = self.rows.get(name)
        return row[0] if row and row[1] > self.clock() else None

    def try_claim(self, name: str, owner: str) -> bool:
        now = self.clock()
        row = self.rows.get(name)
        if row is None or row[1] <= now or row[0] == owner:
            self.rows[name] = (owner, now + self.ttl)
        return self.holder(name) == owner

    def renew(self, name: str, owner: str) -> bool:
        row = self.rows.get(name)
        if row is not None and row[0] == owner:
            self.rows[name] = (owner, self.clock() + self.ttl)
        return self.holder(name) == owner

    def release(self, name: str, owner: str) -> None:
        if self.rows.get(name, ("", None))[0] == owner:
            del self.rows[name]


def new_lock_owner(holder: str) -> str:
    """A unique owner for one notebook run, e.g. ``03-streaming-to-silver:1a2b...``."""
    return f"{holder}:{uuid.uuid4().hex[:12]}"


def acquire_job_locks(
    store,
    names: Iterable[str],
    owner: str,
    *,
    wait_seconds: float,
    poll_seconds: float = JOB_LOCK_POLL_SECONDS,
    sleep: Callable[[float], None] = _time.sleep,
    monotonic: Callable[[], float] = _time.monotonic,
) -> list[str] | None:
    """Claim every lock in ``names`` or none of them.

    Locks are claimed in sorted order, so two jobs that need several locks
    can't deadlock. Retries until ``wait_seconds`` elapse (0 = one attempt)
    and returns the claimed names, or None when another job still holds one.
    """
    ordered = sorted(set(names))
    deadline = monotonic() + max(wait_seconds, 0)
    while True:
        claimed = []
        for name in ordered:
            if not store.try_claim(name, owner):
                break
            claimed.append(name)
        if len(claimed) == len(ordered):
            return claimed
        for name in reversed(claimed):
            store.release(name, owner)
        remaining = deadline - monotonic()
        if remaining <= 0:
            return None
        sleep(min(poll_seconds, remaining))


class JobLocks:
    """A notebook's set of leases, renewed by a heartbeat thread until released."""

    def __init__(
        self,
        store,
        holder: str,
        names: Iterable[str],
        heartbeat_seconds: float = JOB_LOCK_HEARTBEAT_SECONDS,
    ):
        self.store = store
        self.owner = new_lock_owner(holder)
        self.names = sorted(set(names))
        self.heartbeat_seconds = heartbeat_seconds
        self.held: list[str] = []
        self.lost: list[str] = []
        self._stop = threading.Event()
        self._thread: threading.Thread | None = None

    def blockers(self) -> dict[str, str | None]:
        """Current holders of the locks this job needs (for messages)."""
        return {name: self.store.holder(name) for name in self.names}

    def acquire(self, wait_seconds: float, **kwargs) -> bool:
        claimed = acquire_job_locks(
            self.store, self.names, self.owner, wait_seconds=wait_seconds, **kwargs
        )
        if claimed is None:
            return False
        self.held = claimed
        if self.heartbeat_seconds > 0:
            self._thread = threading.Thread(target=self._heartbeat, daemon=True)
            self._thread.start()
        return True

    def _heartbeat(self) -> None:
        while not self._stop.wait(self.heartbeat_seconds):
            self.renew()

    def renew(self) -> None:
        for name in self.held:
            try:
                renewed = self.store.renew(name, self.owner)
            except Exception as exc:
                print(f"Lock {name}: renewal failed ({exc}); retrying at the next heartbeat")
                continue
            if not renewed and name not in self.lost:
                self.lost.append(name)

    def assert_held(self) -> None:
        """Fail before a commit point if a lease expired and another job took it."""
        if self.lost:
            raise RuntimeError(f"Job locks lost before completion: {self.lost}")

    def release(self) -> None:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=self.heartbeat_seconds + 5)
        for name in reversed(self.held):
            self.store.release(name, self.owner)
        self.held = []

In [ ]:
# Hold the Silver writer lock for the whole repair (see job_locks).
JOB_LOCKS = JobLocks(
    DeltaJobLockStore(spark, f"{LAKEHOUSE_NAME}.{SILVER_DB}.{JOB_LOCK_TABLE}"),
    "90-augment-and-dedupe-receipts",
    [SILVER_LOCK],
)
if not JOB_LOCKS.acquire(wait_seconds=LOCK_WAIT_SECONDS):
    raise RuntimeError(f"Silver is locked by another job: {JOB_LOCKS.blockers()}")
# Streaming may have initialized while this run waited for the lock.
assert_streaming_not_initialized()

In [ ]:
def parse_event_ts_from_receipt_id(column_name):
    """Parse RCP{YYYYMMDDHHMM}{store}{sequence} identifiers."""
    ts_string = F.when(
        F.col(column_name).rlike("^RCP[0-9]{12}"),
        F.substring(F.col(column_name), 4, 12),
    )
    return F.to_timestamp(ts_string, "yyyyMMddHHmm")


def schema_signature(frame):
    return tuple(
        (field.name, field.dataType.simpleString())
        for field in frame.schema.fields
    )


def deduplicate_exact_rows(frame, key_columns, context):
    """Collapse exact replays and reject conflicting rows for a business key."""
    null_key = F.col(key_columns[0]).isNull()
    for column_name in key_columns[1:]:
        null_key = null_key | F.col(column_name).isNull()
    if frame.filter(null_key).limit(1).count():
        raise RuntimeError(f"{context} contains a null dedupe key: {key_columns}")

    comparable = (
        frame.drop("__index_level_0__")
        if "__index_level_0__" in frame.columns
        else frame
    )
    distinct_rows = comparable.dropDuplicates()
    conflicts = (
        distinct_rows
        .groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .select(*key_columns)
        .limit(10)
        .collect()
    )
    if conflicts:
        samples = [row.asDict(recursive=True) for row in conflicts]
        raise RuntimeError(
            f"{context}: conflicting rows share dedupe key "
            f"{key_columns}: {samples}"
        )
    return distinct_rows.dropDuplicates(key_columns)


def validate_materialized_table(
    table_name,
    expected_count,
    expected_schema,
    context,
):
    materialized = spark.table(table_name)
    actual_count = materialized.count()
    actual_schema = schema_signature(materialized)
    if actual_count != expected_count or actual_schema != expected_schema:
        raise RuntimeError(
            f"{context} validation failed for {table_name}: "
            f"count {actual_count} != {expected_count} or "
            f"schema {actual_schema} != {expected_schema}"
        )


def stage_and_promote(table_frames):
    """Stage and validate every candidate before replacing any target."""
    staged = []
    for target_table, frame in table_frames:
        table_name = target_table.rsplit(".", 1)[-1]
        candidate_table = (
            f"{LAKEHOUSE_NAME}.{SILVER_STAGE_DB}."
            f"{RUN_TOKEN}__{table_name}__candidate"
        )
        expected_count = frame.count()
        expected_schema = schema_signature(frame)
        (
            frame.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(candidate_table)
        )
        validate_materialized_table(
            candidate_table,
            expected_count,
            expected_schema,
            "Staging",
        )
        staged.append(
            {
                "target": target_table,
                "candidate": candidate_table,
                "expected_count": expected_count,
                "expected_schema": expected_schema,
            }
        )

    promoted = []
    try:
        for item in staged:
            table_name = item["target"].rsplit(".", 1)[-1]
            backup_table = (
                f"{LAKEHOUSE_NAME}.{SILVER_STAGE_DB}."
                f"{RUN_TOKEN}__{table_name}__backup"
            )
            original = spark.table(item["target"])
            item["original_count"] = original.count()
            item["original_schema"] = schema_signature(original)
            item["backup"] = backup_table
            (
                original.write
                .format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .saveAsTable(backup_table)
            )
            validate_materialized_table(
                backup_table,
                item["original_count"],
                item["original_schema"],
                "Backup",
            )

        for item in staged:
            promoted.append(item)
            spark.sql(
                f"CREATE OR REPLACE TABLE {item['target']} "
                f"USING DELTA AS SELECT * FROM {item['candidate']}"
            )
            validate_materialized_table(
                item["target"],
                item["expected_count"],
                item["expected_schema"],
                "Promotion",
            )
    except Exception as exc:
        rollback_errors = []
        for item in reversed(promoted):
            try:
                spark.sql(
                    f"CREATE OR REPLACE TABLE {item['target']} "
                    f"USING DELTA AS SELECT * FROM {item['backup']}"
                )
                validate_materialized_table(
                    item["target"],
                    item["original_count"],
                    item["original_schema"],
                    "Rollback",
                )
            except Exception as rollback_exc:
                rollback_errors.append(
                    f"{item['target']}: {rollback_exc}"
                )
        preserved = [
            item.get("backup", item["candidate"])
            for item in staged
        ] + [item["candidate"] for item in staged]
        if rollback_errors:
            raise RuntimeError(
                f"Promotion failed: {exc}; rollback was incomplete. "
                f"Preserved staging: {preserved}; "
                f"rollback errors: {rollback_errors}"
            ) from exc
        raise RuntimeError(
            f"Promotion failed: {exc}; any changed targets were restored. "
            f"Preserved staging: {preserved}"
        ) from exc

    for item in staged:
        spark.sql(f"DROP TABLE IF EXISTS {item['candidate']}")
        spark.sql(f"DROP TABLE IF EXISTS {item['backup']}")
        print(
            f"Promoted {item['target']} "
            f"({item['expected_count']:,} rows)"
        )

## Step 1 — Augment + dedupe `fact_receipts`

In [ ]:
receipts = spark.read.table(RECEIPTS_TABLE)
print(f"Loaded {RECEIPTS_TABLE} ({receipts.count():,} rows)")
print("Columns before:", receipts.columns)

if "event_ts" not in receipts.columns:
    receipts = receipts.withColumn(
        "event_ts",
        parse_event_ts_from_receipt_id("receipt_id_ext"),
    )
else:
    receipts = receipts.withColumn(
        "event_ts",
        F.coalesce(
            F.col("event_ts"),
            parse_event_ts_from_receipt_id("receipt_id_ext"),
        ),
    )

if "event_date" not in receipts.columns:
    receipts = receipts.withColumn("event_date", F.to_date(F.col("event_ts")))
else:
    receipts = receipts.withColumn(
        "event_date",
        F.coalesce(F.col("event_date"), F.to_date(F.col("event_ts"))),
    )

null_dates = receipts.filter(F.col("event_date").isNull()).count()
print(f"Rows with null event_date after augmentation: {null_dates:,}")

In [ ]:
duplicate_receipt_keys = (
    receipts
    .groupBy("receipt_id_ext")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print(
    "Duplicate receipt_id_ext values before exact-row dedupe: "
    f"{duplicate_receipt_keys:,}"
)

receipts_clean = deduplicate_exact_rows(
    receipts,
    ["receipt_id_ext"],
    "Receipt headers",
)
print(f"Receipt rows after validated dedupe: {receipts_clean.count():,}")

## Step 2 — Propagate `event_ts` / `event_date` to `fact_receipt_lines`

In [ ]:
lines = spark.read.table(LINES_TABLE)
print(f"Loaded {LINES_TABLE} ({lines.count():,} rows)")
print("Columns before:", lines.columns)

header_dates = receipts_clean.select(
    "receipt_id_ext",
    F.col("event_ts").alias("_hdr_event_ts"),
    F.col("event_date").alias("_hdr_event_date"),
)
lines_joined = lines.join(header_dates, on="receipt_id_ext", how="left")

if "event_ts" not in lines.columns:
    lines_aug = lines_joined.withColumn("event_ts", F.col("_hdr_event_ts"))
else:
    lines_aug = lines_joined.withColumn(
        "event_ts",
        F.coalesce(F.col("event_ts"), F.col("_hdr_event_ts")),
    )

if "event_date" not in lines.columns:
    lines_aug = lines_aug.withColumn("event_date", F.col("_hdr_event_date"))
else:
    lines_aug = lines_aug.withColumn(
        "event_date",
        F.coalesce(F.col("event_date"), F.col("_hdr_event_date")),
    )

lines_aug = lines_aug.drop("_hdr_event_ts", "_hdr_event_date")
lines_clean = deduplicate_exact_rows(
    lines_aug,
    ["receipt_id_ext", "line_num"],
    "Receipt lines",
)
missing_line_dates = lines_clean.filter(F.col("event_date").isNull()).count()
print(f"Rows missing event_date after join: {missing_line_dates:,}")
print(f"Receipt line rows after validated dedupe: {lines_clean.count():,}")

## Step 3 — Propagate `event_ts` / `event_date` to `fact_payments`

In [ ]:
payments = spark.read.table(PAYMENTS_TABLE)
print(f"Loaded {PAYMENTS_TABLE} ({payments.count():,} rows)")
print("Columns before:", payments.columns)

payments_joined = payments.join(
    header_dates,
    on="receipt_id_ext",
    how="left",
)

if "event_ts" not in payments.columns:
    payments_aug = payments_joined.withColumn(
        "event_ts",
        F.col("_hdr_event_ts"),
    )
else:
    payments_aug = payments_joined.withColumn(
        "event_ts",
        F.coalesce(F.col("event_ts"), F.col("_hdr_event_ts")),
    )

if "event_date" not in payments.columns:
    payments_aug = payments_aug.withColumn(
        "event_date",
        F.col("_hdr_event_date"),
    )
else:
    payments_aug = payments_aug.withColumn(
        "event_date",
        F.coalesce(F.col("event_date"), F.col("_hdr_event_date")),
    )

payments_aug = payments_aug.drop("_hdr_event_ts", "_hdr_event_date")
payments_clean = deduplicate_exact_rows(
    payments_aug,
    ["transaction_id"],
    "Payments",
)
missing_payment_dates = payments_clean.filter(
    F.col("receipt_id_ext").isNotNull() & F.col("event_date").isNull()
).count()
print(
    "In-store payments missing event_date after join: "
    f"{missing_payment_dates:,}"
)
print(f"Payment rows after validated dedupe: {payments_clean.count():,}")

stage_and_promote(
    [
        (RECEIPTS_TABLE, receipts_clean),
        (LINES_TABLE, lines_clean),
        (PAYMENTS_TABLE, payments_clean),
    ]
)

## Step 3b — Backfill `event_date` on `fact_online_order_headers`

Online order headers always carry `event_ts`, but legacy rows predate the
`event_date` column the semantic model now uses for its `dim_date`
relationship (Direct Lake requires date-to-date relationship columns).


In [ ]:
orders = spark.read.table(ONLINE_ORDERS_TABLE)
print(f"Loaded {ONLINE_ORDERS_TABLE} ({orders.count():,} rows)")
print("Columns before:", orders.columns)

if "event_date" not in orders.columns:
    orders_aug = orders.withColumn(
        "event_date",
        F.to_date(F.col("event_ts")),
    )
else:
    orders_aug = orders.withColumn(
        "event_date",
        F.coalesce(F.col("event_date"), F.to_date(F.col("event_ts"))),
    )

if "__index_level_0__" in orders_aug.columns:
    orders_aug = orders_aug.drop("__index_level_0__")

missing_orders = orders_aug.filter(F.col("event_date").isNull()).count()
print(f"Rows missing event_date after backfill: {missing_orders:,}")
stage_and_promote([(ONLINE_ORDERS_TABLE, orders_aug)])

## Step 3c — Normalize legacy PascalCase columns on promotions / foot traffic / store ops

Early datagen exports wrote PascalCase column names (`PromoCode`, `StoreID`,
`Count`, ...) while the v2 pipeline writes snake_case. Tables loaded from the
legacy exports therefore don't match the semantic model (snake_case per the
project convention), which breaks the Promotions / Foot Traffic / Store
Operations visuals. This step renames legacy columns (coalescing when both
spellings exist), backfills `event_ts` / `event_date`, and drops the pandas
index column.


In [ ]:
LEGACY_RENAMES = {
    f"{LAKEHOUSE_NAME}.{SILVER_DB}.fact_promotions": {
        "EventTS": "event_ts",
        "TraceId": "trace_id",
        "ReceiptId": "receipt_id_ext",
        "PromoCode": "promo_code",
        "DiscountAmount": "discount_amount",
        "DiscountCents": "discount_cents",
        "DiscountType": "discount_type",
        "ProductCount": "product_count",
        "ProductIds": "product_ids",
        "StoreID": "store_id",
        "CustomerID": "customer_id",
    },
    f"{LAKEHOUSE_NAME}.{SILVER_DB}.fact_foot_traffic": {
        "EventTS": "event_ts",
        "TraceId": "trace_id",
        "StoreID": "store_id",
        "SensorId": "sensor_id",
        "Zone": "zone",
        "Dwell": "dwell_seconds",
        "Count": "count",
    },
    f"{LAKEHOUSE_NAME}.{SILVER_DB}.fact_store_ops": {
        "EventTS": "event_ts",
        "TraceId": "trace_id",
        "StoreID": "store_id",
        "OperationTime": "operation_time",
        "OperationType": "operation_type",
    },
}

for table, renames in LEGACY_RENAMES.items():
    try:
        df = spark.read.table(table)
    except AnalysisException:
        print(f"Skipping {table}: table does not exist")
        continue

    print(f"\n=== {table} ===")
    print("Columns before:", df.columns)

    for legacy, snake in renames.items():
        if legacy not in df.columns:
            continue
        if snake in df.columns:
            # both spellings exist (legacy load + v2 streaming): coalesce into snake_case
            df = df.withColumn(snake, F.coalesce(F.col(snake), F.col(legacy))).drop(legacy)
        else:
            df = df.withColumnRenamed(legacy, snake)

    # store_ops legacy rows have operation_time but no event_ts
    if "event_ts" in df.columns and "operation_time" in df.columns:
        df = df.withColumn("event_ts", F.coalesce(F.col("event_ts"), F.col("operation_time")))
    elif "event_ts" not in df.columns and "operation_time" in df.columns:
        df = df.withColumn("event_ts", F.col("operation_time"))

    if "event_date" in df.columns:
        df = df.withColumn("event_date", F.coalesce(F.col("event_date"), F.to_date(F.col("event_ts"))))
    elif "event_ts" in df.columns:
        df = df.withColumn("event_date", F.to_date(F.col("event_ts")))

    if "__index_level_0__" in df.columns:
        df = df.drop("__index_level_0__")

    stage_and_promote([(table, df)])
    print("Columns after:", spark.read.table(table).columns)


## Step 4 — Verification

In [ ]:
for table in (
    RECEIPTS_TABLE,
    LINES_TABLE,
    PAYMENTS_TABLE,
    ONLINE_ORDERS_TABLE,
):
    frame = spark.read.table(table)
    print(f"\n=== {table} ===")
    print("Columns:", frame.columns)
    print("Rows:", frame.count())
    if "event_date" in frame.columns:
        frame.agg(
            F.min("event_date").alias("min_date"),
            F.max("event_date").alias("max_date"),
            F.sum(
                F.when(F.col("event_date").isNull(), 1).otherwise(0)
            ).alias("null_dates"),
        ).show()

dedupe_contracts = {
    RECEIPTS_TABLE: ["receipt_id_ext"],
    LINES_TABLE: ["receipt_id_ext", "line_num"],
    PAYMENTS_TABLE: ["transaction_id"],
}
for table, key_columns in dedupe_contracts.items():
    duplicate_keys = (
        spark.read.table(table)
        .groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    print(f"Duplicate keys remaining in {table}: {duplicate_keys:,}")
    assert duplicate_keys == 0, (
        f"Duplicate keys remain in {table}: {key_columns}"
    )

JOB_LOCKS.release()

print(
    "\nReady. Refresh the retail_model semantic model in Fabric "
    "to pick up the repaired tables."
)